In [16]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [17]:
spark = SparkSession.builder \
    .appName("credit-risk-lakehouse-contrato") \
    .master("spark://spark-master:7077") \
    .getOrCreate()

spark

In [18]:
BASE_PATH = "/app/data"

In [19]:
df_proposta_credito = spark.read.parquet(f"{BASE_PATH}/bronze/proposta_credito")

df_proposta_credito.show(5, truncate=False)

df_proposta_credito.count()

+-----------+----------+---------------+----------------+-----------+-------------+---------------+
|proposta_id|cliente_id|produto        |valor_solicitado|prazo_meses|data_proposta|status_proposta|
+-----------+----------+---------------+----------------+-----------+-------------+---------------+
|1          |1860      |capital_giro   |45929.32        |12         |2025-07-03   |cancelada      |
|2          |1455      |credito_pessoal|95809.27        |48         |2024-11-18   |aprovada       |
|3          |9059      |cartao_credito |34657.56        |24         |2022-11-12   |aprovada       |
|4          |3796      |credito_pessoal|21688.18        |36         |2025-07-15   |aprovada       |
|5          |7758      |cartao_credito |29472.2         |24         |2023-04-14   |reprovada      |
+-----------+----------+---------------+----------------+-----------+-------------+---------------+
only showing top 5 rows



30000

In [20]:
df_propostas_aprovadas = df_proposta_credito \
    .filter(col("status_proposta") == "aprovada")

df_propostas_aprovadas.count()

18027

In [21]:
df_contrato = df_propostas_aprovadas \
    .select(
        monotonically_increasing_id().alias("contrato_id"),
        col("proposta_id"),
        col("cliente_id"),
        col("produto"),
        col("valor_solicitado").alias("valor_contratado"),
        round(rand() * 0.045 + 0.005, 4).alias("taxa_juros"),
        col("prazo_meses"),
        date_add(
            col("data_proposta"),
            floor(rand() * 15 + 1).cast("int")
        ).alias("data_contratacao"),
        element_at(
            array(
                lit("ativo"),
                lit("ativo"),
                lit("ativo"),
                lit("quitado"),
                lit("inadimplente")
            ),
            floor(rand() * 5 + 1).cast("int")
        ).alias("status_contrato")
    )

In [27]:
df_contrato.write \
    .mode("overwrite") \
    .parquet(f"{BASE_PATH}/bronze/contrato")

In [29]:
spark.stop()